# The data agent, step by step

This notebook redoes **by hand, step by step**, what the `fetch_prices` function in `data/data_agent.py` does.
It does not contain the agent's code: it **calls** it. The real code stays in `data/data_agent.py`.

Run the cells one at a time with **Shift + Enter**.

## 0. Setup

Tell Python where the project is (the folder above `notebooks/`), so that `data.data_agent` can be imported.
`autoreload` reloads `data_agent.py` automatically when it changes.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath(".."))

from datetime import date
import pandas as pd
from data.data_agent import load_yahoo_map, yahoo_download, fetch_prices

start, end = date(2025, 1, 1), date.today()

## 1. The mapping table

Belfius gives names (`WORLD`, `NASDAQ 3X`) and ISINs, but Yahoo Finance uses its own symbols (`SXRT.DE`…).
The file `data/etf_yahoo_map.csv` links them. It is created by `python data/check_etfs_yfinance.py`,
which looks for each ETF on the exchange Belfius uses (Xetra, except BEL 20 in Brussels and COPPER LVG / PALLADIUM LVG in Milan).

**Check:** all 50 ETFs are there, each with a symbol and the currency `EUR`.

In [ ]:
ymap = load_yahoo_map()
print(len(ymap), "ETFs in the table")
ymap.head(10)

In [ ]:
# Currencies of the chosen listings: ideally all EUR
ymap["currency"].value_counts()

## 2. Download the raw prices from Yahoo

One call for all 50 ETFs. We take the **real closing prices**, not adjusted for dividends,
because Belfius values the portfolio at the real close and does not credit dividends.

**Result:** one row per day, one column per Yahoo symbol.

In [ ]:
raw = yahoo_download(ymap["yahoo_symbol"].tolist(), start, end)
print(raw.shape[0], "days x", raw.shape[1], "symbols")
raw.head()

## 3. Put all ETFs on the same calendar

Exchanges do not all share the same holidays. All ETFs go on the business-day calendar,
and a day without a quote takes the previous close, **for at most 3 days**.

**Check:** almost no empty cells should remain (except at the very start, for a recently launched ETF).

In [ ]:
calendar = pd.bdate_range(start, end)
aligned = raw.reindex(calendar).ffill(limit=3)

# Days still empty, for the most affected ETFs
aligned.isna().sum().sort_values(ascending=False).head(10)

## 4. Rename with the Belfius names

Everywhere else in the project, an ETF goes by its Belfius name (`WORLD`), never by its Yahoo symbol.

In [ ]:
names = dict(zip(ymap["yahoo_symbol"], ymap["ticker"]))
prices = aligned.rename(columns=names)
prices.tail()

## 5. The final result of `fetch_prices`

The full function does steps 1 to 4 in one go, and returns the table in the format the rest of the team expects:
**one row per day and per ETF**, with the columns `date`, `ticker`, `close_eur`.

**Check:** the same number of ETFs as in step 1, and the same last prices as in step 4.

In [ ]:
df = fetch_prices(start, end)
print(df["ticker"].nunique(), "ETFs,", len(df), "rows")
df.head()

In [ ]:
# The last prices from the function and from step 4 must be identical
comparison = pd.DataFrame({
    "fetch_prices": df.groupby("ticker")["close_eur"].last(),
    "step_4": prices.ffill().iloc[-1].round(4),
})
comparison["gap"] = comparison["fetch_prices"] - comparison["step_4"]
comparison.sort_values("gap", key=abs, ascending=False).head()